# 周期定常の考え方を用いて、自然室温を計算する

In [ ]:
import numpy as np
import pandas as pd

import response_factor as rf

In [ ]:
# 気象データの読み込み
# CSVを読み込み（Shift_JISやCP932など必要に応じて変更）
df = pd.read_csv('weather_data/tokyo.csv', header=None, encoding='ansi')

# 最初の列を項目名、残りを時刻データとして扱う
df_weather = df.set_index(0).T

# 時刻列のインデックス名を付ける（任意）
df_weather.index = [f"i" for i in range(df_weather.shape[0])]

# 風向を削除
df_weather.drop(columns=['風向'], inplace=True)

# すべてのデータをfloat型に変換
df_weather = df_weather.astype(float)

# 外気温度
T_0 = df_weather['乾球温度[℃]'].to_numpy()

# 夜間放射量
rn = df_weather['夜間放射量[W/㎡]'].to_numpy()

# 太陽高度
h = np.radians(df_weather['太陽高度[゜]'].to_numpy())
# 太陽方位角
az = np.radians(df_weather['太陽方位角[゜]'].to_numpy())

# 直達日射量
I_d = df_weather['直達日射量[W/㎡]'].to_numpy()
# 天空日射量
I_s = df_weather['天空日射量[W/㎡]'].to_numpy()
# 地物反射日射量
albedo = 0.1
I_g = (I_d * np.sin(h) + I_s) * albedo

# 結果を表示（または使いたい形式で処理）
df_weather

In [ ]:
# 壁体構成
rs = np.array([
    0.11,
    0.05 / 1.6,
    0.04
])
cs = np.array([
    0.0,
    2000.0 * 0.05 * 1000.0,
    0.0
])

In [ ]:
# 応答係数用パラメータの計算
a0, aa, at, alpha = rf.calc_alpha_matsuo_method(rs=rs, cs=cs, i_max=15)

In [ ]:
# 周期定常応答係数の計算
cyclic_phi_t = rf.calc_cyclic_response_factor(at=at, alpha=alpha, a0=a0, delta_t=3600)
cyclic_phi_a = rf.calc_cyclic_response_factor(at=aa, alpha=alpha, a0=a0, delta_t=3600)

# 放熱応答係数の計算
R_si = 0.11
cyclic_phi_d = cyclic_phi_a * R_si
cyclic_phi_d[0] = 1.0 - cyclic_phi_a[0] * R_si

In [ ]:
# 各種パラメータの計算
c_a = 1005.0
rho_a = 1.2
c_rho_a = c_a * rho_a
delta_t = 3600.0

In [ ]:
# 建物モデル
area = np.array([25.0, 25.0, 20.0, 20.0, 20.0, 20.0])  # 各面の面積 [m^2]
volume = 1000.0  # 建物の体積 [m^3]
delta_t = 3600.0  # 時間刻み [s]
n = 6  # 面の数
# 各面の方位角
azimuth = np.radians(np.array([0, 0, 90, 180, -90, 0]))  # 各面の方位角 [rad]
# 各面の傾斜角
tilt = np.radians(np.array([0, 90, 90, 90, 90, 180]))  # 各面の傾斜角 [rad]

In [ ]:
# 各面の天空に対する形態係数
F_sky = (1.0 + np.cos(tilt)) / 2.0
# 各面の地物に対する形態係数
F_g = 1.0 - F_sky

# 各面の入射角の方向余弦の計算
cos_i = np.sin(h)[:, np.newaxis] * np.cos(tilt)[np.newaxis, :] \
        + np.cos(h)[:, np.newaxis] * np.sin(tilt)[np.newaxis, :] * np.cos(az[:, np.newaxis] - azimuth[np.newaxis, :])
cos_i[cos_i < 0] = 0.0  # 負の値は0にする
print(cos_i)